# 추가 데이터가 왔을 때

이 노트북은 **원본 Excel과 색상 판정 Excel을 CSV로 변환**한 뒤, CSV 합치기 / 재학습 / 저장 모델 적용을 선택합니다.
경로를 입력해야 하므로 `Run All`보다 필요한 칸을 순서대로 실행하세요. 원본 Excel은 바꾸지 않습니다.
먼저 [추가 데이터 가이드](../docs/추가데이터_가이드.md)를 보세요. CSV만 받은 경우 ②를 건너뛰고 ③부터 시작합니다.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path(os.environ.get("ECT_PROJECT_ROOT", str(Path.cwd())))
if not (ROOT / "ectlab").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "ectlab").is_dir():
    raise RuntimeError("프로젝트의 notebooks 폴더에서 열어 주세요.")
sys.path.insert(0, str(ROOT))
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
from IPython.display import display, HTML, FileLink
import pandas as pd
from ectlab.runtime import data_path
from ectlab.data import load_csv, inventory
from ectlab.pipeline import Settings, prepare, distance_analysis, train_models, summarize, export_run
from ectlab.plots import pca_plot, comparison_plot, true_case_plot, frequency_plot
print("실행 준비 완료. 다음 칸을 실행하세요.")

## ① 파일을 넣을 위치
원본은 `data/raw/original`, 기업 회신본은 `data/raw/returned`에 같은 파일명으로 넣습니다. 기존 데이터와 새 데이터의 원본 파일명이 겹치고 시트·행도 같다면 날짜별 시트명으로 구분해야 합니다.

In [ ]:
PRODUCT = os.environ.get("ECT_PRODUCT", "C10A_49513-75270R")
ORIGINAL_EXCEL = ROOT / f"data/raw/original/{PRODUCT}.xlsx"
RETURNED_EXCEL = ROOT / f"data/raw/returned/{PRODUCT}.xlsx"
NEW_CSV = ROOT / f"data/processed/{PRODUCT}_새데이터.csv.gz"
print("원본:", ORIGINAL_EXCEL)
print("기업 회신:", RETURNED_EXCEL)

## ② Excel → CSV 변환
기업이 색상으로 표시한 진성·가성·오류를 읽습니다. 행 번호로만 연결하지 않고 같은 측정값인지 확인합니다. 레이아웃이나 색상 규칙이 바뀌면 담당자에게 확인하세요.

In [ ]:
from ectlab.data import convert_excel
converted = convert_excel(ORIGINAL_EXCEL, RETURNED_EXCEL, NEW_CSV)
display(converted.groupby(["position", "field_label"]).size().rename("측정행 수").reset_index())
print("생성된 CSV:", NEW_CSV)

## ③ 기존 CSV와 추가 CSV 합치기 — 필요할 때만 실행
같은 원본 파일·시트·행·위치가 겹치면 중단합니다. 복사한 데이터나 전체 회신본을 또 추가한 것은 아닌지 확인하세요.

전체 기간을 포함하는 회신본이면 합치지 말고 그 CSV 자체를 ④의 `TRAINING_CSV`로 사용합니다. 기존 라벨을 수정한 회신본 역시 합치지 않고 새 변환본으로 교체합니다.

In [ ]:
MERGE_FILES = False  # 날짜가 다른 추가분만 합칠 때 True
BASE_CSV = data_path(ROOT, "company", PRODUCT)
MERGED_CSV = ROOT / f"data/processed/{PRODUCT}_누적.csv.gz"
if MERGE_FILES:
    base = load_csv(BASE_CSV)
    extra = load_csv(NEW_CSV)
    merged = pd.concat([base, extra], ignore_index=True)
    if merged.id.duplicated().any():
        raise ValueError("원본 파일/시트/행/위치 중복입니다. 전체 회신본을 추가분으로 합치지 마세요.")
    merged.to_csv(MERGED_CSV, index=False, encoding="utf-8-sig")
    print("합친 CSV:", MERGED_CSV)
else:
    print("합치기를 건너뛰었습니다. CSV 한 개를 그대로 사용할 수 있습니다.")

## ④ 추가 데이터로 재학습하기
입력 파일과 날짜를 먼저 바꾸세요. **기존 결과 재현은 원래 날짜 그대로**, **재학습은 새로 정한 날짜**를 사용합니다. 시험용 날짜는 학습 이후로 남겨 둡니다.
새로운 진성이 시험 날짜에 들어오면 그때 진성 검출 여부를 별도로 확인할 수 있습니다.

In [ ]:
TRAINING_CSV = NEW_CSV  # 합친 파일이면 MERGED_CSV
TRAIN_END = "2026-08-20"
CALIBRATION_END = "2026-10-07"
new_df = load_csv(TRAINING_CSV)
settings = Settings(train_end=TRAIN_END, calibration_end=CALIBRATION_END)
packs, counts = prepare(new_df, settings)
display(counts)
distance_result = distance_analysis(packs)
ml_result = train_models(packs)
predictions = pd.concat([distance_result["predictions"], ml_result["predictions"]], ignore_index=True)
display(summarize(predictions))
saved_folder = export_run(ROOT / "results", TRAINING_CSV, settings, packs, distance_result, ml_result)
print("새 학습 모델과 결과:", saved_folder)

## ⑤ 저장된 모델만 적용하기 — 재학습과 별개의 선택 실행
01 또는 ④에서 만든 `models/*.joblib` 중 위치와 모델을 고릅니다. **직접 저장한 모델만** 불러오세요.
이 칸은 학습하지 않습니다. 모델의 위치와 주파수 구성이 같은 새 CSV의 설비 NG만 점수화합니다. 0/노랑·오류·마스터·OK는 ‘분석 대상 아님’으로 남깁니다.

In [ ]:
from ectlab.pipeline import predict_saved
MODEL_PATH = ROOT / "results/여기에_실행폴더명/models/p2_master_proxy_XGBoost.joblib"
INFERENCE_CSV = NEW_CSV
scored = predict_saved(MODEL_PATH, INFERENCE_CSV)
display(scored.head(20))
from datetime import datetime
target = ROOT / "results" / ("새데이터_모델적용_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".csv")
target.parent.mkdir(exist_ok=True)
scored.to_csv(target, index=False, encoding="utf-8-sig")
print("저장:", target)